# AI Prompt Optimizer

## 1. Project Description

Vague prompts like *"Write an advertisement."* give vague results. This notebook sends such a prompt to an LLM
that has been instructed (via a **system prompt**) to act as a prompt engineer. The LLM must answer in a fixed
structure defined by a **Pydantic** model, so the result is always:

1. Original Prompt
2. Improved Prompt
3. Role
4. Context
5. Constraints
6. Expected Output

This shows how **Prompt Engineering + LLM + Pydantic Structured Output** work together. Nothing else is used.

## 2. Imports

In [1]:
# Install the two required packages (only needed the first time)
%pip install -q openai pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q gradio

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from pydantic import BaseModel, Field
from openai import OpenAI

## 3. API / Model Setup

In [ ]:
from getpass import getpass


OPENAI_API_KEY = os.getenv("GROQ_API_KEY") or getpass("Paste your api key here")

MODEL_NAME = "openai/gpt-oss-20b"

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    timeout=30,
    max_retries=1,
)

## 4. Pydantic Output Schema

In [6]:
class OptimizedPrompt(BaseModel):
    original_prompt: str = Field(description="The user's original prompt, unchanged.")
    improved_prompt: str = Field(description="A clearer, complete rewrite of the prompt that keeps the original intent.")
    role: str = Field(description="The role the AI should take when answering (e.g. 'an experienced copywriter').")
    context: str = Field(description="Useful background that helps the AI answer well. Keep it reasonable and relevant.")
    constraints: list[str] = Field(description="A short list of reasonable rules or limits for the answer.")
    expected_output: str = Field(description="A description of the format and content the final answer should have.")

## 5. Optimizer System Prompt

In [7]:
SYSTEM_PROMPT = """
You are an expert prompt engineer. Your job is to take a short or vague user prompt
and turn it into a clear, professional, effective prompt.

Follow these steps:
1. Understand the user's original intent.
2. Identify what information is missing.
3. Improve clarity and wording.
4. Add an appropriate ROLE for the AI.
5. Add useful CONTEXT when possible.
6. Define reasonable CONSTRAINTS (as a short list).
7. Define the EXPECTED OUTPUT format.

Rules:
- Preserve the user's original intent. Do NOT change the task into something different.
  (Example: "Write an advertisement for a coffee shop" must stay about a coffee shop advertisement.)
- Do NOT invent unnecessary requirements or specific facts (names, prices, numbers).
  If details are missing, use neutral placeholders such as [product name] or [target audience].
- Keep every field concise and practical.
- The improved prompt must be a complete prompt that could be sent directly to an AI assistant.
"""

## 6. Prompt Optimization Function

In [8]:
def optimize_prompt(user_prompt: str) -> OptimizedPrompt:
    """Send a simple prompt to the LLM and return a structured OptimizedPrompt."""

    # 1. Call the LLM with the system prompt + the user's prompt,
    #    and tell it to answer using our Pydantic schema.
    completion = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Optimize this prompt:\n\n{user_prompt}"},
        ],
        response_format=OptimizedPrompt,   # <- the structured output / parser step
        temperature=0.3,                   # low = more consistent results
    )

    # 2. The SDK has already parsed and validated the answer into our Pydantic class.
    message = completion.choices[0].message
    if message.parsed is None:
        raise ValueError(f"The model did not return a structured result. Refusal: {message.refusal}")

    result = message.parsed

    # 3. Make sure the original prompt is stored exactly as the user typed it.
    result.original_prompt = user_prompt
    return result

## 7. Helper to Display a Result

In [9]:
def show_result(result: OptimizedPrompt) -> None:
    print("=" * 70)
    print("ORIGINAL PROMPT\n  ", result.original_prompt)
    print("\nIMPROVED PROMPT\n  ", result.improved_prompt)
    print("\nROLE\n  ", result.role)
    print("\nCONTEXT\n  ", result.context)
    print("\nCONSTRAINTS")
    for item in result.constraints:
        print("   -", item)
    print("\nEXPECTED OUTPUT\n  ", result.expected_output)
    print("=" * 70)

## 8. Example Input

In [10]:
example_prompt = "Write an advertisement."

## 9. Run the Optimizer

In [11]:
result = optimize_prompt(example_prompt)

## 10. Display the Structured Result

First as readable text, then as the raw structured data (JSON) to show that the output is a real Pydantic object.

In [12]:
show_result(result)

ORIGINAL PROMPT
   Write an advertisement.

IMPROVED PROMPT
   Write a concise advertisement for [product name] targeting [target audience] in a [desired tone] style. The ad should be suitable for [medium] (e.g., social media post, billboard, radio script) and not exceed [length] words.

ROLE
   an experienced copywriter

CONTEXT
   The advertisement should promote a product or service, targeting a specific audience, with a particular tone, and fit a specified medium.

CONSTRAINTS
   - Maximum 30 words
   - Use active voice
   - Include a call to action
   - Avoid copyrighted material

EXPECTED OUTPUT
   A single paragraph advertisement, no more than 30 words, formatted as plain text.


In [ ]:
# The result is a Pydantic object: access fields directly or convert to JSON
print(type(result))
print(result.role)
print(result.model_dump_json(indent=2))

<class '__main__.OptimizedPrompt'>
an experienced copywriter
{
  "original_prompt": "Write an advertisement.",
  "improved_prompt": "Write an advertisement for [product name] targeting [target audience] in [advertising medium] using a [desired tone] style. Highlight [key benefit] and include a clear call to action. Keep it under 100 words.",
  "role": "an experienced copywriter",
  "context": "This ad is part of a marketing campaign aimed at increasing brand awareness and driving sales.",
  "constraints": [
    "Keep the ad under 100 words.",
    "Use persuasive language.",
    "Avoid industry jargon.",
    "Include a clear call to action."
  ],
  "expected_output": "A single paragraph advertisement of no more than 100 words, following the specified guidelines."
}


## 11. A Few Additional Examples

In [13]:
for prompt in [
    "Explain machine learning.",
    "Create a Python function to read a CSV file.",
    "Write an advertisement for a coffee shop.",   # intent check: must stay about a coffee shop
]:
    show_result(optimize_prompt(prompt))

ORIGINAL PROMPT
   Explain machine learning.

IMPROVED PROMPT
   You are an experienced data scientist. Explain machine learning to a [target audience] in a concise, jargon-free manner. Include a clear definition, key concepts, practical applications, and a simple example. Keep the explanation under 500 words and structure it with headings.

ROLE
   an experienced data scientist

CONTEXT
   The explanation should be suitable for [target audience] and should avoid technical jargon. Use analogies and a simple example to illustrate key ideas.

CONSTRAINTS
   - Keep explanation under 500 words
   - Avoid technical jargon
   - Use analogies
   - Include headings: Definition, Key Concepts, Applications, Example, Summary

EXPECTED OUTPUT
   A structured explanation with headings as specified, written in plain language, under 500 words.
ORIGINAL PROMPT
   Create a Python function to read a CSV file.

IMPROVED PROMPT
   Write a Python function named `read_csv_file` that takes a file path as inp

In [14]:
import inspect
import gradio as gr

def optimize_for_ui(user_prompt):
    """Validate the input, run the existing optimizer, return the five result fields."""
    if not user_prompt or not user_prompt.strip():
        raise gr.Error("Please enter a prompt first.")
    try:
        r = optimize_prompt(user_prompt.strip())
    except Exception as e:
        raise gr.Error(f"Something went wrong: {e}")

    constraints_text = "\n".join(f"• {c}" for c in r.constraints)   # readable list
    return r.improved_prompt, r.role, r.context, constraints_text, r.expected_output


def clear_results():
    """Empty all result boxes when a new optimization starts."""
    return "", "", "", "", ""


def toggle_button(text):
    """The Optimize button is only active when the input is not empty."""
    return gr.Button(interactive=bool(text and text.strip()))


THEME = gr.themes.Base(
    primary_hue="indigo",
    neutral_hue="slate",
    radius_size="lg",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
)

CSS = """
.gradio-container { max-width: 980px !important; margin: 0 auto !important; }

#header { text-align: center; padding: 28px 0 8px; }
#header h1 { font-size: 2.2rem; font-weight: 700; margin: 0 0 6px; }
#header p  { color: var(--body-text-color-subdued); font-size: 1.05rem; margin: 0; }

.card {
    border: 1px solid var(--border-color-primary);
    border-radius: 16px;
    padding: 20px !important;
    background: var(--background-fill-primary);
    box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);
}

#optimize-btn { font-weight: 600; font-size: 1.05rem; }

#result-title { text-align: center; margin-top: 28px; }
#result-title h2 { font-size: 1.4rem; font-weight: 650; margin: 0; }

#footer { text-align: center; color: var(--body-text-color-subdued); font-size: 0.85rem; margin: 24px 0 8px; }

@media (max-width: 640px) {
    #header h1 { font-size: 1.7rem; }
    .card { padding: 14px !important; }
}
"""

style = {"theme": THEME, "css": CSS}
styled_in_blocks = "css" in inspect.signature(gr.Blocks.__init__).parameters
blocks_style = style if styled_in_blocks else {}
launch_style = {} if styled_in_blocks else style

# Copy button on the improved prompt (the argument name differs between Gradio versions)
copy_button = (
    {"show_copy_button": True}
    if "show_copy_button" in inspect.signature(gr.Textbox.__init__).parameters
    else {"buttons": ["copy"]}
)

with gr.Blocks(title="AI Prompt Optimizer", **blocks_style) as demo:

    # Header
    gr.Markdown(
        "# AI Prompt Optimizer\nTurn simple prompts into clear, structured, professional prompts.",
        elem_id="header",
    )

    # Input card
    with gr.Column(elem_classes="card"):
        prompt_input = gr.Textbox(
            label="Your simple prompt",
            placeholder="Example: Write an advertisement for a coffee shop.",
            lines=5,
            max_lines=10,
        )
        with gr.Row():
            clear_button = gr.ClearButton(value="Clear", variant="secondary", scale=1)
            run_button = gr.Button(
                "✨ Optimize Prompt", variant="primary", size="lg",
                interactive=False, elem_id="optimize-btn", scale=3,
            )
        gr.Examples(
            examples=[
                "Write an advertisement.",
                "Explain machine learning.",
                "Create a Python function to read a CSV file.",
            ],
            inputs=prompt_input,
            label="Try an example",
        )

    # Results card
    gr.Markdown("## Optimization Result", elem_id="result-title")
    with gr.Column(elem_classes="card"):
        improved_box = gr.Textbox(label="Improved Prompt", lines=6, max_lines=20, interactive=False, **copy_button)
        with gr.Row():
            role_box = gr.Textbox(label="Role", lines=4, max_lines=10, interactive=False)
            context_box = gr.Textbox(label="Context", lines=4, max_lines=10, interactive=False)
        with gr.Row():
            constraints_box = gr.Textbox(label="Constraints", lines=5, max_lines=15, interactive=False)
            expected_box = gr.Textbox(label="Expected Output", lines=5, max_lines=15, interactive=False)

    gr.Markdown("Prompt Engineering · LLM · Pydantic Structured Output", elem_id="footer")

    outputs = [improved_box, role_box, context_box, constraints_box, expected_box]

    prompt_input.change(toggle_button, inputs=prompt_input, outputs=run_button, queue=False)
    run_button.click(clear_results, outputs=outputs, queue=False).then(optimize_for_ui, inputs=prompt_input, outputs=outputs)
    prompt_input.submit(clear_results, outputs=outputs, queue=False).then(optimize_for_ui, inputs=prompt_input, outputs=outputs)
    clear_button.add([prompt_input] + outputs)

demo.launch(share=True, **launch_style)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Missing file: C:\Users\acer\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\acer\.cache\huggingface\gradio\frpc
